In [7]:
import json
import time
import os

class GameRecorder:
    
    def __init__(self, game):
        self.game = game
        self.moves_history = []
        self.initial_state = {}
        self.is_recording = False
        self.is_replaying = False
        
    def start_recording(self):
        self.is_recording = True
        self.moves_history = []
        
        self.initial_state = {
            'mode': self.game.mode,
            'board_size': self.game.n,
            'player_symbols': self.game.player_symbols.copy(),
            'player_types': self.game.player_type.copy(),
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
        }
        
    def stop_recording(self):
        self.is_recording = False
        
    def record_move(self, row, col, symbol, player):
        if not self.is_recording:
            return
            
        move = {
            'row': row,
            'col': col,
            'symbol': symbol,
            'player': player,
            'timestamp': time.time()
        }
        
        self.moves_history.append(move)
        print(f"Recorded move: {player} placed {symbol} at ({row}, {col})")
        
    def save_to_file(self, filename):
        if not self.moves_history:
            print("No moves to save!")
            return False
            
        try:
            with open(filename, 'w') as f:
                f.write(f"SOS Game Recording\n")
                f.write(f"Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"Mode: {self.initial_state['mode']}\n")
                f.write(f"Board Size: {self.initial_state['board_size']}\n")
                f.write(f"Player Symbols: Blue={self.initial_state['player_symbols']['Blue']}, Red={self.initial_state['player_symbols']['Red']}\n")
                f.write(f"Player Types: Blue={self.initial_state['player_types']['Blue']}, Red={self.initial_state['player_types']['Red']}\n")
                f.write("\n")
                
                f.write("MOVES:\n")
                for i, move in enumerate(self.moves_history):
                    f.write(f"{i+1}. {move['player']} placed {move['symbol']} at position ({move['row']},{move['col']})\n")
                
                if hasattr(self.game, 'game_over') and self.game.game_over:
                    f.write("\nGAME RESULT:\n")
                    if hasattr(self.game, 'winner'):
                        if self.game.winner == 'Draw':
                            f.write("Game ended in a draw.\n")
                        else:
                            f.write(f"Winner: {self.game.winner}\n")
                
                    if self.game.mode == "General" and hasattr(self.game.game_instance, 'scores'):
                        scores = self.game.game_instance.scores
                        f.write(f"Scores: Blue={scores['Blue']}, Red={scores['Red']}\n")
                
            print(f"Game saved to {filename}")
            
            json_filename = filename + ".json"
            self.save_to_json(json_filename)
            
            return True
        except Exception as e:
            print(f"Error saving game: {e}")
            return False
    
    def save_to_json(self, filename):
        game_data = {
            'initial_state': self.initial_state,
            'moves': self.moves_history,
            'final_state': {
                'game_over': self.game.game_over,
                'winner': self.game.winner
            }
        }
        
        if self.game.mode == "General" and hasattr(self.game.game_instance, 'scores'):
            game_data['final_state']['scores'] = self.game.game_instance.scores
        
        try:
            with open(filename, 'w') as f:
                json.dump(game_data, f, indent=2)
            print(f"Game data saved to {filename}")
            return True
        except Exception as e:
            print(f"Error saving game data: {e}")
            return False
    
    def load_from_file(self, filename):
        if filename.endswith('.json'):
            return self.load_from_json(filename)
        elif os.path.exists(filename + ".json"):
            print(f"Found JSON version of the file, loading for replay...")
            return self.load_from_json(filename + ".json")
        else:
            print("No JSON version found, attempting to parse text file...")
            return self.parse_text_file(filename)
    
    def load_from_json(self, filename):

        try:
            with open(filename, 'r') as f:
                game_data = json.load(f)
            if 'initial_state' not in game_data or 'moves' not in game_data:
                print("Invalid game file format")
                return False
                
            self.initial_state = game_data['initial_state']
            self.moves_history = game_data['moves']
            
            print(f"Game loaded from {filename}")
            print(f"Mode: {self.initial_state['mode']}, Board Size: {self.initial_state['board_size']}")
            print(f"Total moves: {len(self.moves_history)}")
            
            self.reset_game()
            
            return True
        except Exception as e:
            print(f"Error loading game from JSON: {e}")
            return False
    
    def parse_text_file(self, filename):
        try:
            with open(filename, 'r') as f:
                lines = f.readlines()
            
            # Parse header information
            mode = "Simple"  # default
            board_size = 8   # default
            player_symbols = {'Blue': 'S', 'Red': 'O'}  # default
            player_types = {'Blue': 'human', 'Red': 'human'}  # default
            
            for line in lines:
                if line.startswith("Mode:"):
                    mode = line.split(":")[1].strip()
                elif line.startswith("Board Size:"):
                    board_size = int(line.split(":")[1].strip())
                elif line.startswith("Player Symbols:"):
                    symbols_part = line.split(":")[1].strip()
                    parts = symbols_part.split(",")
                    for part in parts:
                        if "Blue=" in part:
                            player_symbols['Blue'] = part.split("=")[1].strip()
                        elif "Red=" in part:
                            player_symbols['Red'] = part.split("=")[1].strip()
                elif line.startswith("Player Types:"):
                    types_part = line.split(":")[1].strip()
                    parts = types_part.split(",")
                    for part in parts:
                        if "Blue=" in part:
                            player_types['Blue'] = part.split("=")[1].strip()
                        elif "Red=" in part:
                            player_types['Red'] = part.split("=")[1].strip()
            
            moves_section = False
            moves = []
            
            for line in lines:
                if line.startswith("MOVES:"):
                    moves_section = True
                    continue
                
                if moves_section and line.strip() and not line.startswith("GAME RESULT:"):
                    parts = line.strip().split(" ")
                    player = parts[1]
                    symbol = parts[3]
                    position = parts[6].strip("()")
                    row, col = map(int, position.split(","))
                    
                    moves.append({
                        'row': row,
                        'col': col,
                        'symbol': symbol,
                        'player': player,
                        'timestamp': time.time()
                    })
            
            self.initial_state = {
                'mode': mode,
                'board_size': board_size,
                'player_symbols': player_symbols,
                'player_types': player_types,
                'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
            }
            
            self.moves_history = moves
            
            print(f"Game loaded from text file {filename}")
            print(f"Mode: {mode}, Board Size: {board_size}")
            print(f"Total moves: {len(moves)}")
            
            self.reset_game()
            
            return True
        except Exception as e:
            print(f"Error parsing text file: {e}")
            return False
            
    def reset_game(self):
        try:
            mode = self.initial_state['mode']
            n = self.initial_state['board_size']
            player_symbols = self.initial_state['player_symbols']
            player_types = self.initial_state['player_types']
            
            print(f"Resetting game: {mode} mode, {n}x{n} board")
            print(f"Player symbols: Blue={player_symbols['Blue']}, Red={player_symbols['Red']}")
            
            self.game.__init__(n=n, mode=mode, player_type=player_types)
            self.game.player_symbols = player_symbols
            
            self.game.board = [['' for _ in range(n)] for _ in range(n)]
            self.game.winner = None
            self.game.game_over = False
            
            if hasattr(self.game.game_instance, 'scores'):
                self.game.game_instance.scores = {'Blue': 0, 'Red': 0}
                
            print("Game reset complete")
            return True
        except Exception as e:
            print(f"Error resetting game: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    def replay(self, ui_update_callback=None):
        
        if not self.moves_history:
            print("No moves to replay!")
            return False
        
        print("Starting replay...")
        self.is_replaying = True
        
        try:
            self.reset_game()
            
            if ui_update_callback:
                ui_update_callback()
                time.sleep(0.5)
            
            for i, move in enumerate(self.moves_history):
                row, col, symbol, player = move['row'], move['col'], move['symbol'], move['player']
                
                print(f"Replaying move {i+1}/{len(self.moves_history)}: {player} places {symbol} at ({row}, {col})")
                
                self.game.current_player = player
                
                self.game.board[row][col] = (symbol, player)
                
                if self.game.mode == "General" and hasattr(self.game.game_instance, 'check_score_or_win'):
                    self.game.game_instance.check_score_or_win(row, col)
                elif self.game.mode == "Simple" and hasattr(self.game.game_instance, 'check_score_or_win'):
                    self.game.game_instance.check_score_or_win(row, col)
                else:
                    if player == 'Blue':
                        next_player = 'Red'
                    else:
                        next_player = 'Blue'
                    self.game.current_player = next_player
                
                if ui_update_callback:
                    ui_update_callback()
                    time.sleep(0.5)
            
            print("Replay completed successfully")
            return True
            
        except Exception as e:
            print(f"Error during replay: {e}")
            import traceback
            traceback.print_exc()
            return False
            
        finally:
            self.is_replaying = False